%md
## 🥈 Silver Table — Clean & Fix the Data

&gt; **Rule of Silver:** Remove garbage. Fix types. Keep good rows only.

**Problems in our raw data:**
| Problem | Example | Fix |
|---------|---------|-----|
| Duplicates | Same transaction appears twice | Remove duplicates |
| Bad quantity | `"NULL"`, `"two"`, `-1` | Keep only numbers `>= 1` |
| Bad price | `"N/A"`, `-5.0` | Keep only positive numbers |
| Wrong date format | `"08/15/2024"` | Convert to proper `Date` type |
| Empty payment | `""` (blank string) | Remove rows with empty payment |
| Bad email | `"invalid-email"` | Remove rows with bad email format |

**What we do:**
1. Read from `bronze.sales_raw`
2. Remove duplicate `transaction_id`
3. Cast `quantity` and `unit_price` to numbers
4. Filter out invalid rows
5. Convert `sale_date` to a real date
6. Save clean data to `silver.sales_clean`

In [0]:
# ============================================
# SILVER LAYER: Clean the bronze data
# ============================================

from pyspark.sql.functions import col , to_date , regexp_extract , trim
from pyspark.sql.types import IntegerType , DoubleType 

# Step 1: Read from bronze
df_silver = spark.table("cyntexa_dev.bronze.sales_raw")

# Step 2: Remove duplicate transactions (keep the first one)
df_silver = df_silver.dropDuplicates(["transaction_id"])

# Step 3: Fix data types
df_silver = df_silver \
                .withColumn("quantity" ,col("quantity").try_cast(IntegerType())) \
                .withColumn("unit_price" , col("unit_price").try_cast(DoubleType()))

# Step 4: Convert sale_date from text "MM/dd/yyyy" to a real Date
df_silver = df_silver.withColumn("sale_date" , to_date(col("sale_date") , "MM/dd/yyyy"))

# Step 5: Remove clearly invalid rows
# - quantity must be a number AND >= 1
# - unit_price must be a number AND > 0
# - payment_method cannot be empty or null
# - email must look like a real email (contains @ and .)

# Email validation pattern
email_pattern = r"^[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}$"

df_silver = df_silver.filter(
    (col("quantity").isNotNull()) & (col("quantity") >=1) & 
    (col("unit_price").isNotNull() & (col("unit_price") > 0)) & 
    (trim(col("payment_method")).isNotNull()) & 
    (col("customer_email").rlike(email_pattern))
    )

# Step 6: Drop the ingestion_timestamp (not needed in silver)
df_silver = df_silver.drop("ingestion_timestamp")

# Step 7: Save as silver table
df_silver.write\
         .mode("overwrite")\
         .saveAsTable("cyntexa_dev.silver.sales_clean")

In [0]:
%sql
select COUNT(*) from cyntexa_dev.bronze.sales_raw

In [0]:
%sql
select COUNT(*) from cyntexa_dev.silver.sales_clean 